<a href="https://colab.research.google.com/github/Sweksha-1234/demo-class/blob/main/news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch gradio

In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gradio as gr


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
def generate_fake_news(prompt, max_len, temperature):
    inputs = gpt2_tokenizer(prompt, return_tensors="pt").to(device)

    outputs = gpt2_model.generate(
        **inputs,
        max_length=max_len,
        min_length=80,
        do_sample=True,
        temperature=temperature,
        top_k=50,
        top_p=0.95,
        no_repeat_ngram_size=2,
        repetition_penalty=1.2,
        early_stopping=True
    )

    return gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True)


In [6]:
def detect_news(text):
    inputs = bert_tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = bert_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    predicted_class = torch.argmax(probs, dim=1).item()
    confidence = probs[0][predicted_class].item()

    label = "Fake News" if predicted_class == 0 else "Real News"

    return label, confidence


In [7]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 📰 Interactive Fake News Generator & Detector
    ⚠️ For educational/demo purposes only.
    """)

    with gr.Tab("🛠 Generate News"):
        prompt_input = gr.Textbox(
            label="Enter Headline or Prompt",
            placeholder="e.g. Scientists discover alien life under Antarctica...",
            lines=2
        )

        max_len_slider = gr.Slider(100, 400, value=200, step=10, label="Article Length")
        temp_slider = gr.Slider(0.1, 1.5, value=0.7, step=0.1, label="Creativity (Temperature)")

        generate_btn = gr.Button("Generate Article")
        clear_btn1 = gr.Button("Clear")

        generated_output = gr.Textbox(label="Generated Article", lines=10)

        generate_btn.click(
            generate_fake_news,
            inputs=[prompt_input, max_len_slider, temp_slider],
            outputs=generated_output
        )

        clear_btn1.click(lambda: "", outputs=generated_output)

    with gr.Tab("🔍 Detect News"):
        detect_input = gr.Textbox(
            label="Paste Article to Analyze",
            lines=8
        )

        detect_btn = gr.Button("Analyze")
        clear_btn2 = gr.Button("Clear")

        label_output = gr.Label(label="Prediction")
        confidence_output = gr.Slider(
            0, 1, step=0.01, interactive=False, label="Confidence Score"
        )

        detect_btn.click(
            detect_news,
            inputs=detect_input,
            outputs=[label_output, confidence_output]
        )
        clear_btn2.click(lambda: ("", 0), outputs=[label_output, confidence_output])

/tmp/ipykernel_3670/3205788692.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


In [8]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://34abf5a2f756100be7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
